# Spatial features analysis: morphometrics measurements


### <font color='red'> After clicking on a cell, press "Shift+Enter" to run the code, or click on the "Run" button in the toolbar above.<br>

### Replace "..." signs with the appropriate path to your data.
</font>

In [ ]:
import tifffile
try:
    import napari
    napari_available = True
except ImportError:
    print("Napari is not installed. Some parts of the notebook will not be available.")
    napari_available = False

from pathlib import Path
from tapenade import get_path_to_demo_folder
from tapenade.analysis.deformation.morphometrics import (
        process_genetic_field_cellular,
        smooth_cellular_map,
        process_radial_distance_cellular,
        process_axial_distance_cellular,
        process_cell_density_sigma,
        process_density_gradient,
        process_volume_fraction_sigma,
        process_nuclear_volume_cellular,
        process_nematic_order,
        process_oblate_prolate_cellular,
        process_ellipsoidal_coeff_cellular,
        process_true_strain_maxeig_cellular,
        clear_borders

)
from ensure import check
import numpy as np
from skimage.measure import regionprops

This notebook presents typical use cases of morphometric measurements.

All functions work on 3D (ZYX convention)

Images need to be processed and have a nuclei segmentation and a tissue binary mask, as computed in the prepreocessing notebook.

From there, you can plot maps of :
1. **field of gene expression**
2. **spatial position in the tissue: radial or axial scores**
3. **cell density, cell density gradient magnitude, cell density gradient orientation**
4. **volume fraction**: fraction of space occupied by nuclei **and nuclear volume**
5. **nematic order**: local alignment of nuclei true strain tensors.
6. **nuclei ellipsoidal coefficients, oblate or prolate scores**
7. **nuclei elongation**: true strain maximum eigenvalue of each nuclei

True strain tensors can be computed using the notebook deformation_analysis_notebook.ipynb


Definitions of all metrics are found in Gros, Vanaret, Dunsing-Eichenhauer et al, eLife 2025.


## Visualization

- This notebook computes and plots different 3D maps on napari. Each feature is shown in an independant napari viewer. If you want to plot all of them in one viewer, just comment or uncomment the lines `viewer=napari.Viewer()` (which creates a new viewer) and `napari.run()`
- To adjust the min and max values on the 3D maps, add it as a parameter in the `add_image` function : `viewer.add_image(feature, contrast_limits = [vmin,vmax]`

## 1. Loading the data and choosing parameters (mandatory before computing any feature)

i. Load example data

In [ ]:
path_to_data = get_path_to_demo_folder()

data = tifffile.imread(path_to_data / 'dapi_isotropized.tif')
mask = tifffile.imread(path_to_data / 'mask_def_corr.tif')
labels = tifffile.imread(path_to_data / 'labels_def_corr.tif')

ii. Load your data.

- For simpler interpretation of 3D fields, your mask and labels should have an isotropic pixel size, for example [1,1,1] µm/pix (ZYX)
- Please input 3D data with only one channel

In [ ]:
path_to_data = ...

data = tifffile.imread(Path(path_to_data) / "...")
mask = tifffile.imread(Path(path_to_data) / "...")
labels = tifffile.imread(Path(path_to_data) / "...")

iii. Choose smoothing parameter

In [ ]:
sigma=20

iv. Adjust labels and check shapes

In [ ]:
labels = clear_borders(labels) #removing labels touching the border because their shape is not reliable

props = regionprops(labels)
check(labels.shape).equals(mask.shape).or_raise(
    Exception,
    f"The shape of the selected segmentation {labels.shape} does not match the shape of the mask {mask.shape}. \n",
)


v. Visualize image, mask and segmentation

In [ ]:
viewer = napari.view_image(data, name='raw data')
viewer.add_image(mask, name='mask')
viewer.add_labels(labels, name='labels')

## 2. Field of gene expression

Images of gene expression markers need to be normalized in 3D to remove intensity gradients linked with optical artifacts, see preprocessing_notebook.ipynb

i. Sanity checks on image dimensions and NaN values

In [ ]:
check(data.shape).equals(labels.shape).or_raise(
    Exception,
    f"The shape of the selected data {data.shape} does not match the shape of the labels {labels.shape}. \n",
)
check(data.shape).equals(mask.shape).or_raise(
    Exception,
    f"The shape of the selected data {data.shape} does not match the shape of the mask {mask.shape}. \n",
)

if np.isnan(data).any():
    data[mask == 0] = 0

check(np.isnan(data).any()).equals(False).or_raise(
    Exception,
    "The image contains NaN values. \n",
)


ii. Compute the genetic field

In [ ]:
genetic_field_cellular =  process_genetic_field_cellular(
labels,
data
)
genetic_field_smooth = smooth_cellular_map(
genetic_field_cellular, props, mask, sigma
)

iii. Napari visualization

In [ ]:
viewer=napari.view_labels(labels, name='labels')
viewer.add_image(data)
viewer.add_image(genetic_field_cellular, name='genetic field cellular',colormap='inferno')
viewer.add_image(genetic_field_smooth, name='genetic field smooth',colormap='inferno')

## 3. Spatial position in the tissue

distance to border (radial) or distance along longest axis = AP distance for gastruloids (axial). Designed to be correlated with other maps in the plugin napari-spatial-correlation-plotter

i. Compute cellular maps and smooth them with a gaussian blur


In [ ]:
radial_distance_cellular = process_radial_distance_cellular(
        mask, labels
)
axial_distance_cellular = process_axial_distance_cellular(
        mask, labels
)

axial_distance_smooth = smooth_cellular_map(
        axial_distance_cellular, props, mask, sigma
)
radial_distance_smooth = smooth_cellular_map(
        radial_distance_cellular, props, mask, sigma
)

ii. Napari visualization

In [ ]:
viewer=napari.Viewer()
viewer.add_image(radial_distance_cellular, name='radial distance cellular',colormap='inferno')
viewer.add_image(radial_distance_smooth, name='radial distance smooth',colormap='inferno')
viewer.add_image(axial_distance_cellular, name='axial distance cellular',colormap='inferno')
viewer.add_image(axial_distance_smooth, name='axial distance smooth',colormap='inferno')
napari.run()

## 4. Cell density, cell density gradient magnitude and direction

i. Compute grid to plot cell density gradient

In [ ]:
z_spacing = 10 #to tune
xy_spacing = 10 #to tune

positions_on_grid = (
        np.mgrid[
            [slice(0, labels.shape[0], z_spacing)]
            + [slice(0, dim, xy_spacing) for dim in labels.shape[1:]]
        ]
        .reshape(labels.ndim, -1)
        .T
    )
positions_on_grid = positions_on_grid[
        np.where(
            mask[
                tuple(
                    [
                        positions_on_grid[:, 0],
                        positions_on_grid[:, 1],
                        positions_on_grid[:, 2],
                    ]
                )
            ]
        )
    ]

ii. Compute cell density map and gradient

In [ ]:
# colormap for napari vector fields
def get_napari_angles_cmap():
    cmap_class = napari.utils.colormaps.colormap.Colormap

    colors_strs = [
        (195, 163, 75),
        (163, 103, 44),
        (135, 64, 55),
        (115, 57, 87),
        (92, 83, 139),
        (79, 136, 185),
        (116, 187, 205),
        (180, 222, 198),
        (214, 216, 147),
        (195, 163, 75),
        (163, 103, 44),
        (135, 64, 55),
        (115, 57, 87),
        (92, 83, 139),
        (79, 136, 185),
        (116, 187, 205),
        (180, 222, 198),
        (214, 216, 147),
    ]

    for i, elem in enumerate(colors_strs):
        colors_strs[i] = tuple([x / 255 for x in elem])

    cmap = cmap_class(
        colors_strs,
        controls=np.linspace(0, 1, 1 + len(colors_strs)),
        interpolation="zero",
    )

    return cmap

In [ ]:
density = process_cell_density_sigma(props,mask,sigma)
gradient_field, gradient_magnitude_field,napari_gradient_on_grid, density_gradient_angles = process_density_gradient(
                mask,
    density,
    positions_on_grid,
        )

#normalization of density gradient
lengths = np.linalg.norm(napari_gradient_on_grid[:, 1], axis=1)
napari_gradient_on_grid[:, 1] = (
    napari_gradient_on_grid[:, 1] / np.median(lengths[:, None]) * 5 #*5 is just for visualization purposes
)
napari_gradient_on_grid_proj = napari_gradient_on_grid[:, :, 1:]


iii. Visualize cell density maps

In [ ]:
viewer = napari.Viewer()
viewer.add_image(density,colormap='inferno', name='cell density')
viewer.add_image(gradient_magnitude_field,colormap='inferno', name='density gradient magnitude')
napari.run()

iv. Visualize cell density gradient

In [ ]:
cmap_vectors_gradient = get_napari_angles_cmap()

viewer = napari.Viewer()
viewer.add_vectors(
    napari_gradient_on_grid,
    name="density_gradient",
    properties={"angles": density_gradient_angles},
    vector_style="triangle",
    blending="translucent",
    opacity=1,
    edge_colormap=cmap_vectors_gradient,
    length=5,
    edge_width=5,
)
viewer.add_vectors(
    napari_gradient_on_grid_proj,
    name="density_gradient_proj",
    properties={"angles": density_gradient_angles},
    vector_style="triangle",
    blending="translucent",
    opacity=1,
    edge_colormap=cmap_vectors_gradient,
    length=7,
    edge_width=2,
)
napari.run()

## 5. Volume fraction and nuclear volume

i. Compute volume fraction (space occupied by the nuclei in the mask) as a coarse-grained field, and nuclear volume

In [ ]:
volume_fraction = process_volume_fraction_sigma(
    mask,
    labels,
    sigma
)
nuclear_volume_cellular = process_nuclear_volume_cellular(
    props,mask)
nuclear_volume_smooth = smooth_cellular_map(
    nuclear_volume_cellular, props, mask, sigma
)

ii. Napari visualization

In [ ]:
viewer = napari.Viewer()
viewer.add_image(volume_fraction,colormap='inferno', name='volume fraction')
viewer.add_image(nuclear_volume_cellular,colormap='inferno', name='nuclear volume cellular')
viewer.add_image(nuclear_volume_smooth,colormap='inferno', name='nuclear volume smooth')
napari.run()

## 6. Nematic order

i. Compute orientational order of the nuclei at a local scale.

In [ ]:
nematic_order = process_nematic_order(
    mask,
    labels,
    sigma
)

ii. Napari visualization

In [ ]:
viewer=napari.Viewer()
viewer.add_image(nematic_order, name='nematic order',colormap='inferno')
napari.run()

## 7. Nuclei ellipsoidal coefficients, oblate or prolate scores

i. Two ways of computing it :
- with two different scores, one for oblate, one for prolate (>0)
- with a unique score between -inf and +inf, 0 being spherical or both oblate and prolate

In [ ]:
ellipsoidal_coefficients_oblate, ellipsoidal_coefficients_prolate = process_oblate_prolate_cellular(
    props,mask
)
ellipsoidal_coefficients = process_ellipsoidal_coeff_cellular(
    props, mask
)

ii. Compute min and max values for visualization

In [ ]:
max_oblate_prolate = np.nanmax(np.concatenate((ellipsoidal_coefficients_oblate.flatten(),ellipsoidal_coefficients_prolate.flatten())))
max_extrema_ellipsoidal = max(abs(np.nanmin(ellipsoidal_coefficients.flatten())),abs(np.nanmax(ellipsoidal_coefficients.flatten())))    

In [ ]:
viewer=napari.Viewer()
viewer.add_image(ellipsoidal_coefficients_oblate, name='ellipsoidal_coefficients_oblate',colormap='inferno',contrast_limits=(0,max_oblate_prolate))
viewer.add_image(ellipsoidal_coefficients_prolate, name='ellipsoidal_coefficients_prolate',colormap='inferno',contrast_limits=(0,max_oblate_prolate))
viewer.add_image(ellipsoidal_coefficients, name='ellipsoidal_coefficients',colormap='PiYG',contrast_limits=(-max_extrema_ellipsoidal,max_extrema_ellipsoidal))
napari.run()

## 8. Nuclei elongation

i. Compute true strain maximum eigenvalue to characterize nuclei elongation

In [ ]:
true_strain_maxeig_cellular = process_true_strain_maxeig_cellular(
    props,mask)

true_strain_maxeig_smooth = smooth_cellular_map(
    true_strain_maxeig_cellular, props, mask, sigma)


ii. Napari visualization

In [ ]:
viewer = napari.Viewer()
viewer.add_image(true_strain_maxeig_cellular, name='true_strain_maxeig_cellular',colormap='inferno')
viewer.add_image(true_strain_maxeig_smooth, name='true_strain_maxeig_smooth',colormap='inferno')
napari.run()